# Task 1: News Topic Classifier Using BERT

Fine-tuning BERT on AG News Dataset using Hugging Face Transformers.

In [ ]:
!pip install transformers datasets evaluate accelerate gradio -q

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ag_news")
dataset

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:
!pip install -U transformers datasets accelerate evaluate -q


In [ ]:
# Install compatible versions

!pip uninstall -y transformers torch torchvision torchaudio -q

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

!pip install transformers==4.41.2 accelerate datasets evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 116.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 62.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 99.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 616.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

print("Model Loaded Successfully!")

In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_agnews",
    eval_strategy="epoch",   # old: evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(5000)),
    eval_dataset=tokenized_dataset["test"].shuffle(seed=42).select(range(1000)),
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
!pip install -U transformers datasets accelerate evaluate -q


In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("bert_news_model")
tokenizer.save_pretrained("bert_news_model")


In [ ]:
import gradio as gr
from transformers import pipeline

classifier = pipeline("text-classification", model="bert_news_model")

labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict_news(text):
    result = classifier(text)[0]
    label_id = int(result['label'].split('_')[-1])
    return labels[label_id]

iface = gr.Interface(
    fn=predict_news,
    inputs="text",
    outputs="text",
    title="AG News Topic Classifier"
)

iface.launch()
